# 02 Training

QLoRA fine-tuning on Kaggle T4 (see `docs/claude-3.md` Step 7). Local run requires a CUDA GPU and enough VRAM.

In [ ]:
# ── Colab Setup ────────────────────────────────────────────
import os, torch

# 1. Verify GPU
assert torch.cuda.is_available(), "No GPU! Runtime → Change Runtime Type → T4"
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# 2. Clone repo into Colab working dir
if not os.path.exists('/content/jinyong-finetune'):
    !git clone https://github.com/jxjwilliam/jinyong-finetune.git /content/jinyong-finetune
%cd /content/jinyong-finetune

# 3. Install deps
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    trl==0.10.1 \
    accelerate==0.34.0 \
    bitsandbytes==0.43.3 \
    datasets==2.21.0 \
    pyyaml kaggle

print("Setup complete.")

In [ ]:
# ── Download Jinyong dataset via Kaggle API ─────────────────

# First: upload your kaggle.json
from google.colab import files
print("Upload your kaggle.json now (from kaggle.com → Account → API → Create Token)")
files.upload()   # select kaggle.json from your local machine

# Move it to the right place
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d evilpsycho42/jinyong-wuxia \
    -p /content/jinyong-finetune/data/raw --unzip

!ls -lh /content/jinyong-finetune/data/raw/

In [ ]:
# Verify all critical imports work — ignore gcsfs entirely
import torch
import transformers, peft, trl, accelerate, datasets
import yaml

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"torch        : {torch.__version__}")
print(f"GPU ready    : {torch.cuda.get_device_name(0)}")

### The above the basic setup: bring .ipynb, git clone, set 'T4 GPU', pip install, download datasets

In [ ]:
%cd /content/jinyong-finetune

In [ ]:
!python scripts/clean_text.py --dry-run

In [ ]:
# ── Clean + build instruction pairs ────────────────────────
%cd /content/jinyong-finetune

!python scripts/clean_text.py

!python scripts/build_instructions.py --stats

# Verify
!wc -l data/instructions/jinyong_sft.jsonl

In [ ]:
from pathlib import Path
import os


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts" / "train.py").is_file():
            return cand
    raise FileNotFoundError("Could not find scripts/train.py")


os.chdir(find_repo_root())
print("CWD:", Path.cwd())

## Kaggle: dependencies (uncomment)
```
# !pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 accelerate==0.34.0 bitsandbytes==0.43.3 datasets==2.21.0 pyyaml tqdm torch
# !git clone https://github.com/jxjwilliam/jinyong-finetune.git /kaggle/working/jinyong-finetune
# %cd /kaggle/working/jinyong-finetune
```

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU — enable GPU in Kaggle Settings or use a CUDA machine"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (approx):", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

## Train
Hyperparameters come only from `configs/qlora_config.yaml`.

In [ ]:
import subprocess
import sys

try:
    result = subprocess.run(
        [sys.executable, "scripts/train.py", "--config", "configs/qlora_config.yaml"],
        check=True,
        capture_output=True,
        text=True
    )
    print("STDOUT:", result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error executing train.py: {e}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")

In [ ]:
!pip install accelerate

In [ ]:
!pip install bitsandbytes==0.41.3
!pip install triton
!pip install transformers --upgrade
!pip install accelerate --upgrade

In [ ]:
# Align accelerate with transformers 5.x
!pip install -q -U accelerate
# Then verify
import accelerate, transformers
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")

In [ ]:
# Verify all critical imports work — ignore gcsfs entirely
import torch
import transformers, peft, trl, accelerate, datasets
import yaml

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"torch        : {torch.__version__}")
print(f"GPU ready    : {torch.cuda.get_device_name(0)}")

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/train.py", "--config", "configs/qlora_config.yaml"], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## After training
Adapter weights are written under `outputs/jinyong-qlora/adapter/` (see `training.output_dir` in the YAML). On Kaggle, copy that folder to `/kaggle/working/` if you need it in the session output.